In [2]:
import numpy as np
import pandas as pd
from itertools import combinations
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [5]:
def extract_fields_and_J_matrix(l2_regul):
    
    fields = np.load('parameters_Potts/ByrneT0_fields_%.2e.npy'%l2_regul)
    couplings = np.load('parameters_Potts/ByrneT0_couplings_%.2e.npy'%l2_regul)
    
    J_matrix = np.zeros([21,20,20])
    
    j=0
    for i in combinations(range(7), 2):
        J_matrix[j,:,:] = couplings[i[0],i[1],:,:]
        j+=1
    
    return torch.tensor(fields), torch.tensor(J_matrix)
        
def sorted_log10p_vector_from_fields_and_J_matrix(fields, J_matrix):
    
    lengths = [torch.arange(20, dtype=torch.int8) for i in range(7)]
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(20**7,dtype=torch.float32)

    for i in range(7):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    
    j=0
    for i in combinations(range(7),2):
        p_vector += J_matrix[j,all_seq[:,i[0]],all_seq[:,i[1]]]
        print(i)
        j+=1
    
    p_vector = torch.exp(p_vector)
    Z = p_vector.sum() 
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector).sort()[0] / np.log(10), Z

def log10q_vector_func(data, fields, J_matrix, Z):
    
    # generate p_vector
    q_vector = torch.zeros(len(data),dtype=torch.float32)

    for i in range(7):
        q_vector += fields[i,data[:,i]]
        print(i)
    
    j=0
    for i in combinations(range(7),2):
        q_vector += J_matrix[j,data[:,i[0]],data[:,i[1]]]
        print(i)
        j+=1
        
    q_vector = torch.exp(q_vector)
    
    q_vector /= Z
    print(q_vector.sum())
    
    return np.log10(q_vector.numpy())

In [6]:
data = pd.read_csv('../data/Byrne.csv',index_col=0).query('T0 > 0')
data = data.sort_values('T0', ascending=False)
data = data.iloc[1:]

counts = data['T0'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [7]:
l2_regul = 1e-3
fields, J_matrix = extract_fields_and_J_matrix(l2_regul)

In [8]:
sorted_log10p_vector, Z = sorted_log10p_vector_from_fields_and_J_matrix(fields, J_matrix)

0
1
2
3
4
5
6
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(0, 6)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(3, 4)
(3, 5)
(3, 6)
(4, 5)
(4, 6)
(5, 6)
tensor(1.0000)


In [9]:
log10q_vector = log10q_vector_func(data, fields, J_matrix, Z)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

0
1
2
3
4
5
6
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(0, 6)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(1, 6)
(2, 3)
(2, 4)
(2, 5)
(2, 6)
(3, 4)
(3, 5)
(3, 6)
(4, 5)
(4, 6)
(5, 6)
tensor(0.0320)


In [10]:
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=80, writefolder=False)
df_bins.to_csv('df_bins_Byrne_Potts.csv')

0
tensor(1279999998) tensor(1279864578)
elements in the bin: 135420
nonzeros: 38393
1
tensor(1279864578) tensor(1279666315)
elements in the bin: 198263
nonzeros: 38393
2
tensor(1279666315) tensor(1279420800)
elements in the bin: 245515
nonzeros: 38393
3
tensor(1279420800) tensor(1279129686)
elements in the bin: 291114
nonzeros: 38393
4
tensor(1279129686) tensor(1278795122)
elements in the bin: 334564
nonzeros: 38393
5
tensor(1278795122) tensor(1278422857)
elements in the bin: 372265
nonzeros: 38393
6
tensor(1278422857) tensor(1278005097)
elements in the bin: 417760
nonzeros: 38393
7
tensor(1278005097) tensor(1277549934)
elements in the bin: 455163
nonzeros: 38393
8
tensor(1277549934) tensor(1277049069)
elements in the bin: 500865
nonzeros: 38393
9
tensor(1277049069) tensor(1276500647)
elements in the bin: 548422
nonzeros: 38393
10
tensor(1276500647) tensor(1275909877)
elements in the bin: 590770
nonzeros: 38393
11
tensor(1275909877) tensor(1275268386)
elements in the bin: 641491
nonzer